### Bibliotecas necessárias.

In [7]:
import pandas as pd
import re
import gc

### Tratamento e limpeza dos dados.

In [8]:
class OABCleaner:
    """Classe responsável pela sanitização e formatação de strings e dados."""

    LOWERCASE_EXCEPTIONS = {'e', 'o', 'a', 'de', 'do', 'dos', 'da', 'das'}
    CORRECTIONS = {'tributario': 'tributário'}

    @staticmethod
    def clean_category_name(category: str) -> str:
        # Remove números iniciais e sublinhados
        name = re.sub(r'^\d+_?', '', category).replace('_', ' ')

        # Aplica correções específicas e formatação de título
        words = name.split()
        processed = [
            word.lower() if word.lower() in OABCleaner.LOWERCASE_EXCEPTIONS
            else OABCleaner.CORRECTIONS.get(word.lower(), word).title()
            for word in words
        ]
        return ' '.join(processed)

### Classificar questões.

In [9]:
class OABClassifier:
    """Classe responsável por aplicar métricas de dificuldade e metadados."""

    @staticmethod
    def count_laws(text: str) -> int:
        if not text: return 0
        return len(re.findall(r'\b(lei|leis)\b', str(text).lower()))

    @classmethod
    def apply_difficulty(cls, df: pd.DataFrame) -> pd.DataFrame:
        def get_level(text):
            n = cls.count_laws(text)
            if n == 0: return 1 # Estagiário
            if n <= 2: return 2 # Analista
            if n <= 4: return 3 # Juiz
            return 4            # Ministro

        df['level'] = df['choices'].apply(get_level)
        return df

### Carregar questões e linha guia.

In [27]:
class OABDataLoader:
    """Classe responsável pelo carregamento e processamento dos datasets."""

    BASE_URL = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/main/data/oab_bench"
    EDITIONS_MAP = {
        '37': 2023, '38': 2023, '39': 2023,
        '40': 2024, '41': 2024, '42': 2024,
        '43': 2025, '44': 2025, '45': 2025
    }

    def __init__(self):
        self.cleaner = OABCleaner()
        self.classifier = OABClassifier()

    def _fetch_jsonl(self, url: str) -> pd.DataFrame:
        return pd.read_json(url, lines=True)

    def load_questions(self) -> pd.DataFrame:
        url = f"{self.BASE_URL}/question.jsonl"
        df = self._fetch_jsonl(url)
        df['category'] = df['category'].apply(self.cleaner.clean_category_name)
        df['edition'] = df['question_id'].apply(lambda x: x[:2])
        df['year'] = df['edition'].map(self.EDITIONS_MAP).fillna('Desconhecido')
        df['turns'] = df['turns'].apply(lambda x: x[0] if isinstance(x, list) and x else '')
        return df

    def load_guidelines(self) -> pd.DataFrame:
        url = f"{self.BASE_URL}/reference_answer/guidelines.jsonl"
        df = self._fetch_jsonl(url)
        df['choices'] = df['choices'].apply(lambda x: x[0]['turns'][0] if isinstance(x, list) and x else '')
        return df

    def get_combined_dataset(self, apply_levels=True) -> pd.DataFrame:
        df_q = self.load_questions()
        df_g = self.load_guidelines()

        # Merge
        merged = pd.merge(df_q, df_g[['question_id', 'choices']], on='question_id', how='inner')

        # Seleção de colunas conforme original
        columns = ['question_id', 'edition', 'year', 'statement', 'turns', 'category', 'system', 'choices']
        merged = merged[columns]

        if apply_levels:
            merged = self.classifier.apply_difficulty(merged)

        merged.insert(0, 'num', range(1, len(merged) + 1))
        merged['type'] = 'Aberta'
        return merged

    @staticmethod
    def save_to_jsonl(df: pd.DataFrame, filename: str):
        try:
            df.to_json(filename, orient='records', lines=True, force_ascii=False)
            print(f"Sucesso: {filename} salvo.")
        except Exception as e:
            print(f"Erro ao salvar: {e}")

### Execução

In [28]:
# Original code from dC2HYrojjs7Q
loader = OABDataLoader()
df_questions = loader.get_combined_dataset()

# Exemplo: Pegando o subconjunto de 31 a 40
df_my_questions = df_questions.iloc[30:40].copy()

# Exportar
loader.save_to_jsonl(df_questions, 'open_questions.jsonl')

Sucesso: open_questions.jsonl salvo.


### Visualização pontual. Descomentar sobre demanda.

In [29]:
# df_my_questions

,num,question_id,edition,year,statement,turns,category,system,choices,level,type
30,31,39_direito_tributario_peca_profissional,39,2023,"PEÇA PRÁTICO-PROFISSIONAL\n\nPedro de Camões, ...",,Direito Tributário,Você é um bacharel em direito que está realiza...,"A medida judicial cabível é a ação anulatória,...",4,Aberta
31,32,39_direito_tributario_questao_1,39,2023,QUESTÃO\n\nA sociedade empresária Metalúrgica ...,"A partir de quando se deve contar, no caso con...",Direito Tributário,Você é um bacharel em direito que está realiza...,O prazo para oferta dos embargos à execução fi...,2,Aberta
32,33,39_direito_tributario_questao_2,39,2023,QUESTÃO\n\nA sociedade empresária Tudo Verde L...,A qual dos municípios o ISS de jardinagem é de...,Direito Tributário,Você é um bacharel em direito que está realiza...,O ISS de jardinagem é devido ao Município Beta...,1,Aberta
33,34,39_direito_tributario_questao_3,39,2023,QUESTÃO\n\nA Administração Tributária Federal ...,É válida a exigência de depósito do montante c...,Direito Tributário,Você é um bacharel em direito que está realiza...,"Não, pois é inconstitucional a exigência de de...",1,Aberta
34,35,39_direito_tributario_questao_4,39,2023,QUESTÃO\n\nA sociedade empresária Faz Tudo Ltd...,Está correto o argumento da necessidade de not...,Direito Tributário,Você é um bacharel em direito que está realiza...,"Não está correto, porque a entrega de declaraç...",1,Aberta
35,36,40_direito_administrativo_peca_profissional,40,2024,PEÇA PRÁTICO-PROFISSIONAL\n\nO Ministério Públ...,,Direito Administrativo,Você é um bacharel em direito que está realiza...,O(a) examinando(a) deve apresentar recurso de ...,4,Aberta
36,37,40_direito_administrativo_questao_1,40,2024,QUESTÃO\n\nA sociedade empresária Sagaz S.A. e...,Há necessidade de demonstração do elemento sub...,Direito Administrativo,Você é um bacharel em direito que está realiza...,Não. A responsabilização administrativa por at...,3,Aberta
37,38,40_direito_administrativo_questao_2,40,2024,QUESTÃO\n\nDeterminada informação de interesse...,É lícita a cobrança efetuada pelo órgão respon...,Direito Administrativo,Você é um bacharel em direito que está realiza...,Não. A submissão e o processamento de pedido d...,3,Aberta
38,39,40_direito_administrativo_questao_3,40,2024,QUESTÃO\n\nCerta Secretaria do Estado Alfa fez...,É possível a utilização do sistema de registro...,Direito Administrativo,Você é um bacharel em direito que está realiza...,Sim. A Administração poderá contratar a execuç...,2,Aberta
39,40,40_direito_administrativo_questao_4,40,2024,"QUESTÃO\n\nRecentemente, Iná foi aprovada em c...",A aprovação de Iná no mencionado concurso impo...,Direito Administrativo,Você é um bacharel em direito que está realiza...,"Não. Iná foi aprovada para emprego público, ao...",1,Aberta
